In [ ]:
#| hide
import random
from edgeslm.coffee import *

# edgeslm

> Small language models for narrow, domain-specific tasks at the edge: synthetic data, LoRA fine-tuning, evaluation and GGUF export

edgeslm fine-tunes small language models (SLMs) for narrow, domain-specific tasks, and runs them on edge hardware such as a Raspberry Pi 5 or an Arduino UNO Q. A large hosted model can do the same tasks, but every call costs money and energy, data leaves the site, and the service depends on a provider and a network. A narrow task that repeats every day uses only a small part of a large model's expertise, so a small model trained for it can do the job locally.

The pipeline is domain-neutral: a task is defined by data, namely its input format, its output format and its examples. It goes from synthetic inputs and outputs written by a large teacher model, through LoRA fine-tuning and evaluation, to a GGUF model that llama.cpp runs on the board. The approach is described, and refined as the project goes, in [Scope](https://franckalbinet.github.io/edgeslm/design/scope.html).


## The first task

The first task comes from [speaking-plants](https://fr.anckalbi.net/speaking-plants/), which detects water stress in coffee plants with sensor nodes. Each day, code in the domain project produces a *triage* for each farm: which nodes to watch or fix, how severe each problem is, and what the data shows about its cause, such as whether water still reaches the soil. The SLM turns the triage into a short WhatsApp message for the farmer, with an action for each problem.

The real triage does not exist yet, so [`coffee`](https://franckalbinet.github.io/edgeslm/coffee.html) generates synthetic records that follow the triage contract in the scope, using the tuple sampler in [`synth`](https://franckalbinet.github.io/edgeslm/synth.html). Here is one record:

In [ ]:
triage(dict(problem='water stress', pattern='cluster', severity='moderate'), random.Random(0))

{'farm': 'Fazenda São João',
 'date': '2026-05-17',
 'nodes': 45,
 'items': [{'problem': 'water stress',
   'location': 'Block B, drip line B1',
   'pattern': 'cluster',
   'severity': 'moderate',
   'since': 'since Friday',
   'soil_water': 'arriving',
   'nodes_flagged': 6,
   'nodes_in_group': 7,
   'amount': 'almost all'}]}

A message for this record could read:

> ⚠️ Block B, drip line B1: almost all monitored plants have shown water stress since Friday, although their soil still gets water. Check the plants on this line for heat damage or root problems.

## Status

The project is building a proof of concept for the first task. The scope, the synthetic triage records, the code checks, the teacher prompt and the training data are done. Claude Haiku wrote the training messages.

The [baseline](05_baseline.ipynb) and [fine-tuning](06_finetune.ipynb) experiments compare Qwen3.5 0.8B and 2B on 270 test records. Untuned, with rules and two examples in the prompt, they write messages that pass every check for 123 and 70 records. Fine-tuned on 100 examples, with a one-line prompt, both pass for 255 or more. The 0.8B fine-tuned on 199 examples passes for 264.

The next step is to convert the fine-tuned 0.8B to GGUF and run it with llama.cpp on an Arduino UNO Q, to measure speed, memory and quality on the board.

## Install


Install the latest version from GitHub:

```sh
pip install git+https://github.com/franckalbinet/edgeslm.git
```
